# MMI Laser Scan Analysis

This notebook follows the folder rules for this project:
- beginner-level notebook
- easy to check by eye
- simple code with enough comments
- all analysis kept inside the notebook


## What this notebook does

1. Load the laser scan data.
2. Find local peaks and local dips for `gc1->gc2` and `gc1->gc3`.
3. Calculate insertion loss (IL) at those marker points.
4. Calculate extinction ratio (ER) at each dip.
5. Plot four subplots for manual review.

In this notebook, **dip** means local minimum.


## ER rule used here

For each dip:
1. Find the nearest peak on the left.
2. Find the nearest peak on the right.
3. Convert the two peak values from dB to linear optical power.
4. Average those two peak powers in linear scale.
5. Compare that average peak power with the dip power.

If a dip does not have one peak on both sides, that ER point is skipped.


In [ ]:
from pathlib import Path
import io
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import find_peaks

plt.style.use("seaborn-v0_8-whitegrid")


## Step 1: Load the data file

The original measurement file has a text header and an embedded ZIP block.
The function below reads both parts and returns the data arrays.


In [ ]:
data_files = sorted(Path('.').glob('*.loglaserscan_ymlz'))

if not data_files:
    raise FileNotFoundError('No .loglaserscan_ymlz file was found in this folder.')

DATA_FILE = data_files[0]
CHANNELS = ['gc1->gc2', 'gc1->gc3']


def load_loglaserscan_ymlz(file_path):
    """Read the header and data arrays from one measurement file."""
    raw_bytes = Path(file_path).read_bytes()
    zip_start = raw_bytes.find(b'PK\x03\x04')

    if zip_start == -1:
        raise ValueError('Could not find the embedded ZIP data in the measurement file.')

    header_text = raw_bytes[:zip_start].decode('utf-8', errors='replace')

    wavelength_match = re.search(
        r'wav: # xdata\s+first: ([0-9.]+)\s+last: ([0-9.]+)\s+points: (\d+)',
        header_text,
    )

    if wavelength_match is None:
        raise ValueError('Could not read the wavelength axis from the file header.')

    wavelength_nm = np.linspace(
        float(wavelength_match.group(1)),
        float(wavelength_match.group(2)),
        int(wavelength_match.group(3)),
    )

    scan_data = {'wavelength_nm': wavelength_nm}

    with zipfile.ZipFile(io.BytesIO(raw_bytes[zip_start:])) as zip_file:
        for file_name in zip_file.namelist():
            if file_name.endswith('.npy'):
                channel_name = file_name[:-4]
                scan_data[channel_name] = np.load(io.BytesIO(zip_file.read(file_name)))

    return header_text, scan_data


header_text, scan_data = load_loglaserscan_ymlz(DATA_FILE)
wavelength_nm = scan_data['wavelength_nm']

print(f'Using data file: {DATA_FILE.name}')

file_summary = pd.DataFrame(
    {
        'Channel': CHANNELS,
        'Point count': [len(scan_data[channel]) for channel in CHANNELS],
        'Min transmission (dB)': [scan_data[channel].min() for channel in CHANNELS],
        'Max transmission (dB)': [scan_data[channel].max() for channel in CHANNELS],
    }
)

file_summary


## Step 2: Set simple analysis parameters

If the markers do not look right, start by changing these two values.


In [ ]:
# Minimum marker strength in dB.
prominence_db = 1.0

# Minimum gap between two nearby markers, in data points.
min_distance_points = 250


## Step 3: Find peaks, dips, IL, and ER

Important notes:
- IL is stored as a positive loss number, so `IL = -transmission_dB`.
- ER is only calculated at dip locations.
- ER uses the average of the left and right peak powers in linear scale.


In [ ]:
def db_to_linear_power(power_db):
    """Convert power from dB to linear scale."""
    return 10 ** (power_db / 10)


def linear_power_to_db(power_linear):
    """Convert power from linear scale to dB."""
    return 10 * np.log10(power_linear)


def build_marker_table(channel_name, wavelength_nm, trace_db, marker_index, marker_type):
    """Create a small table that is easy to inspect manually."""
    return pd.DataFrame(
        {
            'Channel': channel_name,
            'Marker type': marker_type,
            'Index': marker_index,
            'Wavelength (nm)': wavelength_nm[marker_index],
            'Transmission (dB)': trace_db[marker_index],
            'IL (dB)': -trace_db[marker_index],
        }
    )


def analyze_channel(channel_name, wavelength_nm, trace_db, prominence_db, min_distance_points):
    """Find markers and calculate ER using the project rule."""

    # Find local high points.
    peak_index, _ = find_peaks(trace_db, prominence=prominence_db, distance=min_distance_points)

    # Find local low points by searching peaks on the negative trace.
    dip_index, _ = find_peaks(-trace_db, prominence=prominence_db, distance=min_distance_points)

    peak_table = build_marker_table(channel_name, wavelength_nm, trace_db, peak_index, 'peak')
    dip_table = build_marker_table(channel_name, wavelength_nm, trace_db, dip_index, 'dip')

    er_rows = []
    skipped_dip_count = 0

    for dip in dip_index:
        # The ER rule needs one peak on the left and one peak on the right.
        left_peaks = peak_index[peak_index < dip]
        right_peaks = peak_index[peak_index > dip]

        if len(left_peaks) == 0 or len(right_peaks) == 0:
            skipped_dip_count += 1
            continue

        left_peak = left_peaks[-1]
        right_peak = right_peaks[0]

        left_peak_db = trace_db[left_peak]
        right_peak_db = trace_db[right_peak]
        dip_db = trace_db[dip]

        left_peak_linear = db_to_linear_power(left_peak_db)
        right_peak_linear = db_to_linear_power(right_peak_db)
        average_peak_linear = (left_peak_linear + right_peak_linear) / 2
        dip_linear = db_to_linear_power(dip_db)

        er_value_db = linear_power_to_db(average_peak_linear / dip_linear)

        er_rows.append(
            {
                'Channel': channel_name,
                'Dip index': dip,
                'Dip wavelength (nm)': wavelength_nm[dip],
                'Dip transmission (dB)': dip_db,
                'Dip IL (dB)': -dip_db,
                'Left peak wavelength (nm)': wavelength_nm[left_peak],
                'Left peak transmission (dB)': left_peak_db,
                'Right peak wavelength (nm)': wavelength_nm[right_peak],
                'Right peak transmission (dB)': right_peak_db,
                'Average peak power (linear)': average_peak_linear,
                'ER (dB)': er_value_db,
            }
        )

    er_table = pd.DataFrame(er_rows)

    if er_table.empty:
        average_er_db = np.nan
        er_wavelength_nm = np.array([])
        er_db = np.array([])
    else:
        average_er_db = er_table['ER (dB)'].mean()
        er_wavelength_nm = er_table['Dip wavelength (nm)'].to_numpy()
        er_db = er_table['ER (dB)'].to_numpy()

    return {
        'peak_index': peak_index,
        'dip_index': dip_index,
        'peak_table': peak_table,
        'dip_table': dip_table,
        'er_table': er_table,
        'er_wavelength_nm': er_wavelength_nm,
        'er_db': er_db,
        'skipped_dip_count': skipped_dip_count,
        'average_peak_il_db': peak_table['IL (dB)'].mean(),
        'average_dip_il_db': dip_table['IL (dB)'].mean(),
        'average_er_db': average_er_db,
    }


results = {}

for channel_name in CHANNELS:
    results[channel_name] = analyze_channel(
        channel_name=channel_name,
        wavelength_nm=wavelength_nm,
        trace_db=scan_data[channel_name],
        prominence_db=prominence_db,
        min_distance_points=min_distance_points,
    )

summary_table = pd.DataFrame(
    {
        'Channel': CHANNELS,
        'Peak count': [len(results[channel]['peak_index']) for channel in CHANNELS],
        'Dip count': [len(results[channel]['dip_index']) for channel in CHANNELS],
        'ER point count': [len(results[channel]['er_db']) for channel in CHANNELS],
        'Skipped dip count': [results[channel]['skipped_dip_count'] for channel in CHANNELS],
        'Average peak IL (dB)': [results[channel]['average_peak_il_db'] for channel in CHANNELS],
        'Average dip IL (dB)': [results[channel]['average_dip_il_db'] for channel in CHANNELS],
        'Average ER (dB)': [results[channel]['average_er_db'] for channel in CHANNELS],
    }
)

summary_table


## Step 4: Manual check tables

These small tables are included so the marker finding and ER rule are easy to inspect without reading the whole code.


In [ ]:
print('gc1->gc2 peaks')
display(results['gc1->gc2']['peak_table'].head(8))

print('gc1->gc2 dips')
display(results['gc1->gc2']['dip_table'].head(8))

print('gc1->gc2 ER points')
display(results['gc1->gc2']['er_table'].head(8))


In [ ]:
print('gc1->gc3 peaks')
display(results['gc1->gc3']['peak_table'].head(8))

print('gc1->gc3 dips')
display(results['gc1->gc3']['dip_table'].head(8))

print('gc1->gc3 ER points')
display(results['gc1->gc3']['er_table'].head(8))


## Step 5: Plot the result

The final figure has four subplots:
1. Raw data.
2. `gc1->gc2` with peak and dip markers.
3. `gc1->gc3` with peak and dip markers.
4. ER plotted as a line with markers.


In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=False)
ax_raw, ax_gc2, ax_gc3, ax_er = axes.flat

# Plot 1: the two raw traces.
for channel_name in CHANNELS:
    ax_raw.plot(wavelength_nm, scan_data[channel_name], linewidth=1.2, label=channel_name)

ax_raw.set_title('Raw data')
ax_raw.set_xlabel('Wavelength (nm)')
ax_raw.set_ylabel('Transmission (dB)')
ax_raw.legend()

# Plot 2 and 3: raw data with marker locations.
channel_axes = {'gc1->gc2': ax_gc2, 'gc1->gc3': ax_gc3}

for channel_name, axis in channel_axes.items():
    trace_db = scan_data[channel_name]
    channel_result = results[channel_name]

    axis.plot(wavelength_nm, trace_db, color='0.75', linewidth=1.0, label='Raw data')
    axis.scatter(
        wavelength_nm[channel_result['peak_index']],
        trace_db[channel_result['peak_index']],
        s=24,
        color='tab:red',
        label='Local peaks',
        zorder=3,
    )
    axis.scatter(
        wavelength_nm[channel_result['dip_index']],
        trace_db[channel_result['dip_index']],
        s=24,
        color='tab:blue',
        label='Local dips',
        zorder=3,
    )
    axis.set_title(channel_name)
    axis.set_xlabel('Wavelength (nm)')
    axis.set_ylabel('Transmission (dB)')
    axis.legend(fontsize=8)

# Plot 4: ER points connected by a line.
for channel_name in CHANNELS:
    ax_er.plot(
        results[channel_name]['er_wavelength_nm'],
        results[channel_name]['er_db'],
        marker='o',
        markersize=4,
        linewidth=1.5,
        label=channel_name,
    )

ax_er.set_title('Extinction ratio (ER)')
ax_er.set_xlabel('Wavelength (nm)')
ax_er.set_ylabel('ER (dB)')
ax_er.legend()

figure.suptitle(f'MMI analysis for {DATA_FILE.name}', fontsize=14)
figure.tight_layout()

output_figure = DATA_FILE.with_name(f'{DATA_FILE.stem}_analysis_summary.png')
figure.savefig(output_figure, dpi=200, bbox_inches='tight')
print(f'Saved figure: {output_figure.name}')

plt.show()
